# CLT Cascade Classifier — Full Training Run

Trains BERT-base and RoBERTa-base (Stage 1 salience + Stage 2 CLT dimensions) on GPU.
Outputs are saved to Google Drive after each run so a Colab disconnect loses nothing.

**Run cells in order on first use. Each cell is independently re-runnable after a reconnect.**

| Cell | What it does |
|------|--------------|
| 1 | GPU check — hard-stops if no GPU |
| 2 | Mount Google Drive |
| 3 | Clone repo + install dependencies |
| 4 | Upload data files + verify counts |
| 5 | BERT Stage 1 training |
| 6 | BERT Stage 2 training |
| 7 | RoBERTa Stage 1 training |
| 8 | RoBERTa Stage 2 training |
| 9 | Results summary table |
| 10 | Commit RoBERTa configs to GitHub |

## Cell 1 — GPU Check

Verifies a GPU is attached. Hard-stops if not — training on CPU is not viable.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("No GPU detected.")
    print("Switch to GPU runtime: Runtime > Change runtime type > T4 GPU, then re-run all cells.")
    raise RuntimeError("GPU required. Switch runtime and reconnect.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU detected: {gpu_name}  ({vram_gb:.1f} GB)")
print(f"torch version: {torch.__version__}")
print("Cell 1 OK.")

## Cell 2 — Mount Google Drive

Creates `/content/drive/MyDrive/CLT_Thesis/outputs/` for persistent checkpoint storage.
Re-running this cell after a reconnect is safe.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"Drive output folder: {DRIVE_OUTPUT_DIR}")
print("Contents:", os.listdir(DRIVE_OUTPUT_DIR) or "(empty)")
print("Cell 2 OK.")

## Cell 3 — Clone Repo + Install Dependencies

- Sets `HF_HOME` so model weights are cached under `/content/hf_cache/` (not the default)
- Clones the repo if it does not already exist (safe to re-run after reconnect)
- Installs pinned dependencies from `requirements.txt`

In [ ]:
import os
os.environ["HF_HOME"] = "/content/hf_cache"

# Clone repo
!git clone https://github.com/KrisHoffmann/Bachelor-Thesis-IK.git /content/Bachelor-Thesis-IK
%cd /content/Bachelor-Thesis-IK

# Install only what Colab does not already have
# torch, numpy, sklearn are pre-installed — do NOT reinstall them
!pip install -q "transformers>=4.41.0" accelerate datasets krippendorff pyyaml

print("\nRepo contents:")
!ls -la
print("\nReady to proceed to Cell 4.")

## Cell 4 — Upload Data Files

Upload `train.json`, `dev.json`, and `test.json` when prompted.
The cell will verify counts (723 / 176 / 155) and hard-stop on any mismatch.

**Do not skip this cell.** Re-runnable after reconnect — files already in `data/processed/` are detected and the upload step is skipped.

In [ ]:
import os
import shutil
import sys

sys.path.insert(0, "/content/Bachelor-Thesis-IK")
os.makedirs("data/processed", exist_ok=True)

REQUIRED = ["train.json", "dev.json", "test.json"]
already_present = all(os.path.exists(f"data/processed/{f}") for f in REQUIRED)

if already_present:
    print("Data files already present in data/processed/ — skipping upload.")
else:
    print("Upload train.json, dev.json, test.json now — do this before continuing")
    print("(Use the file chooser that appears below)")
    from google.colab import files
    uploaded = files.upload()  # blocks until user selects files

    if not uploaded:
        raise RuntimeError("No files uploaded. Re-run this cell and upload all three JSON files.")

    for fname, content in uploaded.items():
        dest = f"data/processed/{fname}"
        # files.upload() writes to CWD; move to data/processed/
        if os.path.exists(fname) and not os.path.exists(dest):
            shutil.move(fname, dest)
        elif os.path.exists(fname):
            shutil.move(fname, dest)
        print(f"  Saved: {dest}")

    missing = [f for f in REQUIRED if not os.path.exists(f"data/processed/{f}")]
    if missing:
        raise RuntimeError(
            f"Missing files after upload: {missing}\n"
            "Re-run this cell and upload all three JSON files."
        )

# Verify counts — hard-fail on mismatch
from src.data import verify_splits
try:
    verify_splits("data/processed")
except AssertionError as e:
    raise RuntimeError(f"Data verification failed: {e}") from e

print("Data verified. Ready to train.")

## Cell 5 — BERT-base Stage 1 Training

Fine-tunes `bert-base-uncased` for salience classification (binary, class-weighted).
Config: `configs/bert_stage1.yaml` — 5 epochs, batch 16, lr 2e-5.
Outputs saved to `outputs/bert_stage1/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "bert_stage1"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
METRICS_PATH = f"{LOCAL_OUT}/metrics.json"

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 1 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", "configs/bert_stage1.yaml", "--stage", "1"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best dev macro-F1
with open(METRICS_PATH) as f:
    m = json.load(f)
print(f"\nBERT Stage 1 — Dev macro-F1: {m['macro_f1']:.4f}  |  Accuracy: {m['accuracy']:.4f}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"BERT Stage 1 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 6 — BERT-base Stage 2 Training

Trains four parallel 3-class CLT heads (Temporal / Spatial / Social / Hypothetical)
on the 621 salient training sentences.
Config: `configs/bert_stage2.yaml` — 5 epochs, batch 16, lr 2e-5, warmup 10%.
Outputs saved to `outputs/bert_stage2/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "bert_stage2"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
METRICS_PATH = f"{LOCAL_OUT}/epoch_metrics.json"

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 2 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", "configs/bert_stage2.yaml", "--stage", "2"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best epoch metrics (highest mean_macro_f1)
with open(METRICS_PATH) as f:
    epoch_records = json.load(f)

best = max(epoch_records, key=lambda r: r["mean_macro_f1"])
print(f"\nBERT Stage 2 — Best epoch {best['epoch']}  |  Mean macro-F1: {best['mean_macro_f1']:.4f}")
for dim in ("temporal", "spatial", "social", "hypothetical"):
    f1 = best[dim]["macro_f1"]
    collapsed = best[dim].get("collapsed", False)
    note = "  [COLLAPSE]" if collapsed else ""
    print(f"  {dim:<14}: macro-F1 = {f1:.4f}{note}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"BERT Stage 2 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 7 — RoBERTa-base Stage 1 Training

Creates `configs/roberta_stage1.yaml` (copy of bert_stage1.yaml with `roberta-base`)
and runs the same Stage 1 pipeline. The training script is reused unchanged.
Outputs saved to `outputs/roberta_stage1/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

import yaml

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "roberta_stage1"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
ROBERTA_CFG = "configs/roberta_stage1.yaml"
METRICS_PATH = f"{LOCAL_OUT}/metrics.json"

# Create RoBERTa Stage 1 config (derived from bert_stage1.yaml)
with open("configs/bert_stage1.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model_name"] = "roberta-base"
cfg["output_dir"] = f"outputs/{RUN_NAME}"
cfg["run_name"] = RUN_NAME
with open(ROBERTA_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Created {ROBERTA_CFG}")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  RoBERTa Stage 1 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", ROBERTA_CFG, "--stage", "1"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best dev macro-F1
with open(METRICS_PATH) as f:
    m = json.load(f)
print(f"\nRoBERTa Stage 1 — Dev macro-F1: {m['macro_f1']:.4f}  |  Accuracy: {m['accuracy']:.4f}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"RoBERTa Stage 1 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 8 — RoBERTa-base Stage 2 Training

Creates `configs/roberta_stage2.yaml` (copy of bert_stage2.yaml with `roberta-base`)
and runs the same Stage 2 pipeline.
Outputs saved to `outputs/roberta_stage2/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

import yaml

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "roberta_stage2"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
ROBERTA_CFG = "configs/roberta_stage2.yaml"
METRICS_PATH = f"{LOCAL_OUT}/epoch_metrics.json"

# Create RoBERTa Stage 2 config (derived from bert_stage2.yaml)
with open("configs/bert_stage2.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model_name"] = "roberta-base"
cfg["output_dir"] = f"outputs/{RUN_NAME}"
cfg["run_name"] = RUN_NAME
with open(ROBERTA_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Created {ROBERTA_CFG}")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  RoBERTa Stage 2 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", ROBERTA_CFG, "--stage", "2"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best epoch metrics
with open(METRICS_PATH) as f:
    epoch_records = json.load(f)

best = max(epoch_records, key=lambda r: r["mean_macro_f1"])
print(f"\nRoBERTa Stage 2 — Best epoch {best['epoch']}  |  Mean macro-F1: {best['mean_macro_f1']:.4f}")
for dim in ("temporal", "spatial", "social", "hypothetical"):
    f1 = best[dim]["macro_f1"]
    collapsed = best[dim].get("collapsed", False)
    note = "  [COLLAPSE]" if collapsed else ""
    print(f"  {dim:<14}: macro-F1 = {f1:.4f}{note}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"RoBERTa Stage 2 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 9 — Results Summary Table

Loads metrics from all four completed runs and prints a comparison table.
Missing runs are skipped with a warning rather than crashing.
This table goes directly into the thesis.

In [ ]:
import json
import os

DIMS = ("temporal", "spatial", "social", "hypothetical")

def load_stage1_metrics(run_name):
    path = f"outputs/{run_name}/metrics.json"
    if not os.path.exists(path):
        # Also check Drive as fallback
        path = f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}/metrics.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def load_stage2_metrics(run_name):
    path = f"outputs/{run_name}/epoch_metrics.json"
    if not os.path.exists(path):
        path = f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}/epoch_metrics.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        records = json.load(f)
    return max(records, key=lambda r: r["mean_macro_f1"])

def fmt(val):
    return f"{val:.3f}" if val is not None else "  N/A "

rows = []
for model_label, s1_run, s2_run in [
    ("BERT-base",  "bert_stage1",    "bert_stage2"),
    ("RoBERTa-base", "roberta_stage1", "roberta_stage2"),
]:
    s1 = load_stage1_metrics(s1_run)
    s2 = load_stage2_metrics(s2_run)

    if s1 is None:
        print(f"WARNING: Stage 1 metrics not found for {model_label} ({s1_run}) — run skipped.")
    if s2 is None:
        print(f"WARNING: Stage 2 metrics not found for {model_label} ({s2_run}) — run skipped.")

    s1_f1   = s1["macro_f1"] if s1 else None
    s2_temp = s2["temporal"]["macro_f1"] if s2 else None
    s2_spat = s2["spatial"]["macro_f1"] if s2 else None
    s2_soc  = s2["social"]["macro_f1"] if s2 else None
    s2_hypo = s2["hypothetical"]["macro_f1"] if s2 else None
    s2_mean = s2["mean_macro_f1"] if s2 else None

    rows.append((model_label, s1_f1, s2_temp, s2_spat, s2_soc, s2_hypo, s2_mean))

# Print table
header = f"{'Model':<14} | {'S1 macro-F1':>11} | {'S2 Temporal':>11} | {'S2 Spatial':>10} | {'S2 Social':>9} | {'S2 Hypothetical':>15} | {'S2 Mean':>7}"
sep = "-" * len(header)
print(sep)
print(header)
print(sep)
for model_label, s1_f1, s2_temp, s2_spat, s2_soc, s2_hypo, s2_mean in rows:
    print(
        f"{model_label:<14} | {fmt(s1_f1):>11} | {fmt(s2_temp):>11} | {fmt(s2_spat):>10} | "
        f"{fmt(s2_soc):>9} | {fmt(s2_hypo):>15} | {fmt(s2_mean):>7}"
    )
print(sep)
print("\nNote: Stage 2 Temporal and Spatial macro-F1 may be low due to ~90% N/A class imbalance.")
print("      COLLAPSE means the model predicted only one class for that dimension.")

## Cell 10 — Commit RoBERTa Configs to GitHub

Commits `configs/roberta_stage1.yaml` and `configs/roberta_stage2.yaml` to the repo.
If the push fails with an authentication error, step-by-step PAT instructions are printed.

In [ ]:
import subprocess

def run_git(args, check=True):
    result = subprocess.run(["git"] + args, capture_output=True, text=True)
    return result

# Confirm the two config files exist before committing
import os
for cfg_file in ["configs/roberta_stage1.yaml", "configs/roberta_stage2.yaml"]:
    if not os.path.exists(cfg_file):
        raise FileNotFoundError(
            f"{cfg_file} not found — run Cells 7 and 8 first to generate the RoBERTa configs."
        )

# Set git identity (Colab has no global git config by default)
run_git(["config", "user.email", "colab-training@thesis"])
run_git(["config", "user.name", "Colab Training Run"])

# Stage the two new config files only
r = run_git(["add", "configs/roberta_stage1.yaml", "configs/roberta_stage2.yaml"])
if r.returncode != 0:
    raise RuntimeError(f"git add failed: {r.stderr}")

# Check if there is actually something to commit
r = run_git(["diff", "--cached", "--name-only"])
staged = r.stdout.strip()
if not staged:
    print("Nothing new to commit — RoBERTa configs already committed.")
else:
    print(f"Staged files:\n{staged}")
    r = run_git(["commit", "-m", "feat: add RoBERTa configs from Colab training run"])
    if r.returncode != 0:
        raise RuntimeError(f"git commit failed: {r.stderr}")
    print("Committed.")

# Push
r = run_git(["push", "origin", "main"])
if r.returncode == 0:
    print("Pushed to origin/main successfully.")
else:
    stderr = r.stderr
    if any(word in stderr.lower() for word in ["authentication", "auth", "403", "username", "password", "token"]):
        print("Push failed: GitHub authentication required.")
        print()
        print("To push from Colab, set up a Personal Access Token (PAT):")
        print("  1. Go to https://github.com/settings/tokens")
        print("  2. Generate new token (classic) — tick the 'repo' scope.")
        print("  3. Copy the token (shown only once).")
        print("  4. In a new Colab cell, run:")
        print("       from google.colab import userdata")
        print("       import subprocess")
        print("       token = userdata.get('GITHUB_TOKEN')  # or paste directly")
        print("       subprocess.run(['git', 'remote', 'set-url', 'origin',")
        print("           f'https://{token}@github.com/KrisHoffmann/Bachelor-Thesis-IK.git'])")
        print("       subprocess.run(['git', 'push', 'origin', 'main'])")
        print()
        print("Commit is saved locally — nothing is lost. Push when auth is configured.")
    else:
        print(f"Push failed with unexpected error:\n{stderr}")
        print("Commit is saved locally. Investigate the error above before retrying.")

## Cell 11 — Test-Set Evaluation of Shootout Checkpoints (dev-trained models)

> **Note: this is the second time the test set is used.**
> The first was the train+dev→test final-run evaluation (Final Cells A–E).
> This second use evaluates the dev-trained shootout checkpoints on test for direct
> dev-vs-test comparison, per supervisor request.
> **Test set was not used to tune any hyperparameter.**

Loads the four original shootout checkpoints (trained on `train.json` only) and
evaluates each on the held-out `test.json`. No retraining is performed.

| Sub-cell | Model | Stage | Checkpoint |
|----------|-------|-------|------------|
| 11a | BERT-base | Stage 1 | `bert_stage1/checkpoint-138/` |
| 11b | BERT-base | Stage 2 | `bert_stage2/best_model.pt` |
| 11c | RoBERTa-base | Stage 1 | `roberta_stage1/checkpoint-138/` |
| 11d | RoBERTa-base | Stage 2 | `roberta_stage2/best_model.pt` |

Outputs are saved locally and backed up to Drive.

### Cell 11a — BERT Stage 1 on Test

Loads `bert_stage1/checkpoint-138/` (dev-trained shootout checkpoint) and evaluates
on `test.json`. Saves `metrics.json` and `confusion_matrix.png` to
`outputs/bert_stage1_test_eval/`. Also backs up to Drive.

In [ ]:
import json
import os
import shutil
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import precision_score, recall_score

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

from src.data import load_split
from src.evaluate import compute_metrics_dict
from src.model import build_model

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
CKPT_LOCAL = "outputs/bert_stage1/checkpoint-138"
CKPT_DRIVE = f"{DRIVE_OUTPUT_DIR}/bert_stage1/checkpoint-138"
OUT_LOCAL  = "outputs/bert_stage1_test_eval"
OUT_DRIVE  = f"{DRIVE_OUTPUT_DIR}/bert_stage1_test_eval"
TEST_PATH  = "data/processed/test.json"
MAX_LENGTH = 128

# Resolve checkpoint — prefer local, fall back to Drive
ckpt_path = CKPT_LOCAL if os.path.isdir(CKPT_LOCAL) else CKPT_DRIVE
if not os.path.isdir(ckpt_path):
    raise FileNotFoundError(
        f"BERT Stage 1 checkpoint not found at {CKPT_LOCAL} or {CKPT_DRIVE}.\n"
        "Run Cell 5 first, or ensure the checkpoint is on Drive."
    )
print(f"Loading checkpoint: {ckpt_path}")

os.makedirs(OUT_LOCAL, exist_ok=True)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model     = build_model(ckpt_path)
model.to(device).eval()

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

test_ds = load_split(TEST_PATH).map(tokenize, batched=True)
test_ds = test_ds.remove_columns(["text"])
test_ds.set_format("torch")
print(f"Test set: {len(test_ds)} sentences")

collator = DataCollatorWithPadding(tokenizer)
loader   = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collator)

all_preds, all_gold = [], []
with torch.no_grad():
    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
        ).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        all_gold.extend(batch["labels"].tolist())

# Core metrics via evaluate.py
core = compute_metrics_dict(all_preds, all_gold)

# Precision and recall per class (not emitted by evaluate.py)
preds_arr = np.array(all_preds)
gold_arr  = np.array(all_gold)
per_class_precision = precision_score(
    gold_arr, preds_arr, average=None, labels=[0, 1], zero_division=0
).tolist()
per_class_recall = recall_score(
    gold_arr, preds_arr, average=None, labels=[0, 1], zero_division=0
).tolist()

metrics_out = {
    "macro_f1":            core["macro_f1"],
    "accuracy":            core["accuracy"],
    "per_class_precision": per_class_precision,
    "per_class_recall":    per_class_recall,
    "per_class_f1":        core["per_class_f1"],
    "confusion_matrix":    core["confusion_matrix"].tolist(),
    "krippendorff_alpha":  core["krippendorff_alpha"],
}

with open(f"{OUT_LOCAL}/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
with open(f"{OUT_LOCAL}/test_predictions.json", "w") as f:
    json.dump({"predictions": all_preds, "gold": all_gold}, f)

# Confusion matrix PNG (winning model)
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    disp = ConfusionMatrixDisplay(
        confusion_matrix=core["confusion_matrix"],
        display_labels=["not salient", "salient"],
    )
    disp.plot(colorbar=False)
    plt.title("Test confusion matrix — BERT Stage 1 (dev-trained)")
    plt.tight_layout()
    plt.savefig(f"{OUT_LOCAL}/confusion_matrix.png", dpi=150)
    plt.close()
    print("Confusion matrix PNG saved.")
except ImportError:
    print("matplotlib not installed — skipping confusion matrix PNG")

# Back up to Drive
if os.path.isdir(OUT_DRIVE):
    shutil.rmtree(OUT_DRIVE)
shutil.copytree(OUT_LOCAL, OUT_DRIVE)

print("\n" + "=" * 60)
print("BERT Stage 1 — TEST evaluation (dev-trained checkpoint)")
print("-" * 60)
print(f"  Macro-F1 : {metrics_out['macro_f1']:.4f}")
print(f"  Accuracy : {metrics_out['accuracy']:.4f}")
print(f"  Kripp. α : {metrics_out['krippendorff_alpha']:.4f}")
print(f"  Precision: not_salient={per_class_precision[0]:.4f}  salient={per_class_precision[1]:.4f}")
print(f"  Recall   : not_salient={per_class_recall[0]:.4f}  salient={per_class_recall[1]:.4f}")
print(f"  F1       : not_salient={core['per_class_f1'][0]:.4f}  salient={core['per_class_f1'][1]:.4f}")
print("  Confusion matrix (rows=gold, cols=pred):")
for row in metrics_out["confusion_matrix"]:
    print(f"    {row}")
print("=" * 60)
print(f"Outputs: {OUT_LOCAL}/  |  Drive: {OUT_DRIVE}/")


### Cell 11b — BERT Stage 2 on Test

Loads `bert_stage2/best_model.pt` and evaluates on the salient sentences of `test.json`.
Saves `metrics.json` and one confusion matrix PNG per dimension
(labelled `["N/A", "Near", "Far"]`) to `outputs/bert_stage2_test_eval/`.

In [ ]:
import json
import os
import shutil
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import precision_score, recall_score

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

from src.data import DIMS, INV_LABEL_MAP, load_split_stage2
from src.evaluate import evaluate_stage2
from src.model import CLTStage2Model

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
CKPT_LOCAL = "outputs/bert_stage2/best_model.pt"
CKPT_DRIVE = f"{DRIVE_OUTPUT_DIR}/bert_stage2/best_model.pt"
CFG_LOCAL  = "outputs/bert_stage2/config.yaml"
CFG_DRIVE  = f"{DRIVE_OUTPUT_DIR}/bert_stage2/config.yaml"
OUT_LOCAL  = "outputs/bert_stage2_test_eval"
OUT_DRIVE  = f"{DRIVE_OUTPUT_DIR}/bert_stage2_test_eval"
TEST_PATH  = "data/processed/test.json"
MAX_LENGTH = 128
MODEL_NAME = "bert-base-uncased"

# Resolve checkpoint
ckpt_path = CKPT_LOCAL if os.path.exists(CKPT_LOCAL) else CKPT_DRIVE
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f"BERT Stage 2 checkpoint not found at {CKPT_LOCAL} or {CKPT_DRIVE}.\n"
        "Run Cell 6 first."
    )
print(f"Loading checkpoint: {ckpt_path}")

# Resolve model_name from saved config if available
cfg_path = CFG_LOCAL if os.path.exists(CFG_LOCAL) else (
    CFG_DRIVE if os.path.exists(CFG_DRIVE) else None
)
if cfg_path:
    import yaml
    with open(cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    MODEL_NAME = _cfg.get("model_name", MODEL_NAME)
print(f"Encoder base: {MODEL_NAME}")

os.makedirs(OUT_LOCAL, exist_ok=True)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = CLTStage2Model(MODEL_NAME).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

def tokenize_s2(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

test_ds       = load_split_stage2(TEST_PATH).map(tokenize_s2, batched=True)
test_ds_torch = test_ds.remove_columns(["text"])
test_ds_torch.set_format("torch")
print(f"Salient test sentences: {len(test_ds_torch)}")

collator = DataCollatorWithPadding(tokenizer)
loader   = DataLoader(test_ds_torch, batch_size=32, shuffle=False, collate_fn=collator)

all_preds = {d: [] for d in DIMS}
all_gold  = {d: [] for d in DIMS}
with torch.no_grad():
    for batch in loader:
        logits_dict = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
        )
        for dim in DIMS:
            pred_idx = torch.argmax(logits_dict[dim], dim=-1).cpu().tolist()
            gold_idx = batch[dim].tolist()
            all_preds[dim].extend(INV_LABEL_MAP[p] for p in pred_idx)
            all_gold[dim].extend(INV_LABEL_MAP[g] for g in gold_idx)

core = evaluate_stage2(all_preds, all_gold)

# Build serialisable metrics dict with per-class precision/recall added
metrics_out = {"mean_macro_f1": core["mean_macro_f1"]}
for dim in DIMS:
    dm        = core[dim]
    preds_arr = np.array(all_preds[dim])
    gold_arr  = np.array(all_gold[dim])
    per_class_precision = precision_score(
        gold_arr, preds_arr, average=None, labels=[-1, 0, 1], zero_division=0
    ).tolist()
    per_class_recall = recall_score(
        gold_arr, preds_arr, average=None, labels=[-1, 0, 1], zero_division=0
    ).tolist()
    metrics_out[dim] = {
        "macro_f1":            dm["macro_f1"],
        "per_class_precision": per_class_precision,
        "per_class_recall":    per_class_recall,
        "per_class_f1":        dm["per_class_f1"],
        "confusion_matrix":    dm["confusion_matrix"].tolist(),
        "krippendorff_alpha":  dm["krippendorff_alpha"],
        "collapsed":           dm["collapsed"],
    }

with open(f"{OUT_LOCAL}/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
with open(f"{OUT_LOCAL}/test_predictions.json", "w") as f:
    json.dump({"predictions": all_preds, "gold": all_gold}, f, indent=2)

# Confusion matrix PNGs — one per dimension
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    for dim in DIMS:
        cm   = core[dim]["confusion_matrix"]
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=["N/A", "Near", "Far"]
        )
        disp.plot(colorbar=False)
        plt.title(f"Test confusion matrix — BERT Stage 2 {dim} (dev-trained)")
        plt.tight_layout()
        plt.savefig(f"{OUT_LOCAL}/confusion_matrix_{dim}.png", dpi=150)
        plt.close()
    print("Confusion matrix PNGs saved.")
except ImportError:
    print("matplotlib not installed — skipping confusion matrix PNGs")

# Back up to Drive
if os.path.isdir(OUT_DRIVE):
    shutil.rmtree(OUT_DRIVE)
shutil.copytree(OUT_LOCAL, OUT_DRIVE)

print("\n" + "=" * 60)
print("BERT Stage 2 — TEST evaluation (dev-trained checkpoint)")
print("-" * 60)
print(f"  Mean macro-F1 : {core['mean_macro_f1']:.4f}")
for dim in DIMS:
    dm   = metrics_out[dim]
    note = "  [COLLAPSE]" if dm["collapsed"] else ""
    print(f"  {dim:<14}: macro-F1={dm['macro_f1']:.4f}  α={dm['krippendorff_alpha']:.4f}{note}")
print("=" * 60)
print(f"Outputs: {OUT_LOCAL}/  |  Drive: {OUT_DRIVE}/")


### Cell 11c — RoBERTa Stage 1 on Test

Loads `roberta_stage1/checkpoint-138/` (dev-trained shootout checkpoint) and evaluates
on `test.json`. Saves `metrics.json` and `confusion_matrix.png` to
`outputs/roberta_stage1_test_eval/`.

In [ ]:
import json
import os
import shutil
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import precision_score, recall_score

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

from src.data import load_split
from src.evaluate import compute_metrics_dict
from src.model import build_model

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
CKPT_LOCAL = "outputs/roberta_stage1/checkpoint-138"
CKPT_DRIVE = f"{DRIVE_OUTPUT_DIR}/roberta_stage1/checkpoint-138"
OUT_LOCAL  = "outputs/roberta_stage1_test_eval"
OUT_DRIVE  = f"{DRIVE_OUTPUT_DIR}/roberta_stage1_test_eval"
TEST_PATH  = "data/processed/test.json"
MAX_LENGTH = 128

# Resolve checkpoint — prefer local, fall back to Drive
ckpt_path = CKPT_LOCAL if os.path.isdir(CKPT_LOCAL) else CKPT_DRIVE
if not os.path.isdir(ckpt_path):
    raise FileNotFoundError(
        f"RoBERTa Stage 1 checkpoint not found at {CKPT_LOCAL} or {CKPT_DRIVE}.\n"
        "Run Cell 7 first, or ensure the checkpoint is on Drive."
    )
print(f"Loading checkpoint: {ckpt_path}")

os.makedirs(OUT_LOCAL, exist_ok=True)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model     = build_model(ckpt_path)
model.to(device).eval()

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

test_ds = load_split(TEST_PATH).map(tokenize, batched=True)
test_ds = test_ds.remove_columns(["text"])
test_ds.set_format("torch")
print(f"Test set: {len(test_ds)} sentences")

collator = DataCollatorWithPadding(tokenizer)
loader   = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collator)

all_preds, all_gold = [], []
with torch.no_grad():
    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
        ).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        all_gold.extend(batch["labels"].tolist())

# Core metrics via evaluate.py
core = compute_metrics_dict(all_preds, all_gold)

# Precision and recall per class (not emitted by evaluate.py)
preds_arr = np.array(all_preds)
gold_arr  = np.array(all_gold)
per_class_precision = precision_score(
    gold_arr, preds_arr, average=None, labels=[0, 1], zero_division=0
).tolist()
per_class_recall = recall_score(
    gold_arr, preds_arr, average=None, labels=[0, 1], zero_division=0
).tolist()

metrics_out = {
    "macro_f1":            core["macro_f1"],
    "accuracy":            core["accuracy"],
    "per_class_precision": per_class_precision,
    "per_class_recall":    per_class_recall,
    "per_class_f1":        core["per_class_f1"],
    "confusion_matrix":    core["confusion_matrix"].tolist(),
    "krippendorff_alpha":  core["krippendorff_alpha"],
}

with open(f"{OUT_LOCAL}/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
with open(f"{OUT_LOCAL}/test_predictions.json", "w") as f:
    json.dump({"predictions": all_preds, "gold": all_gold}, f)

# Confusion matrix PNG
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    disp = ConfusionMatrixDisplay(
        confusion_matrix=core["confusion_matrix"],
        display_labels=["not salient", "salient"],
    )
    disp.plot(colorbar=False)
    plt.title("Test confusion matrix — RoBERTa Stage 1 (dev-trained)")
    plt.tight_layout()
    plt.savefig(f"{OUT_LOCAL}/confusion_matrix.png", dpi=150)
    plt.close()
    print("Confusion matrix PNG saved.")
except ImportError:
    print("matplotlib not installed — skipping confusion matrix PNG")

# Back up to Drive
if os.path.isdir(OUT_DRIVE):
    shutil.rmtree(OUT_DRIVE)
shutil.copytree(OUT_LOCAL, OUT_DRIVE)

print("\n" + "=" * 60)
print("RoBERTa Stage 1 — TEST evaluation (dev-trained checkpoint)")
print("-" * 60)
print(f"  Macro-F1 : {metrics_out['macro_f1']:.4f}")
print(f"  Accuracy : {metrics_out['accuracy']:.4f}")
print(f"  Kripp. α : {metrics_out['krippendorff_alpha']:.4f}")
print(f"  Precision: not_salient={per_class_precision[0]:.4f}  salient={per_class_precision[1]:.4f}")
print(f"  Recall   : not_salient={per_class_recall[0]:.4f}  salient={per_class_recall[1]:.4f}")
print(f"  F1       : not_salient={core['per_class_f1'][0]:.4f}  salient={core['per_class_f1'][1]:.4f}")
print("  Confusion matrix (rows=gold, cols=pred):")
for row in metrics_out["confusion_matrix"]:
    print(f"    {row}")
print("=" * 60)
print(f"Outputs: {OUT_LOCAL}/  |  Drive: {OUT_DRIVE}/")


### Cell 11d — RoBERTa Stage 2 on Test

Loads `roberta_stage2/best_model.pt` and evaluates on the salient sentences of `test.json`.
Saves `metrics.json` and one confusion matrix PNG per dimension
(labelled `["N/A", "Near", "Far"]`) to `outputs/roberta_stage2_test_eval/`.

In [ ]:
import json
import os
import shutil
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import precision_score, recall_score

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

from src.data import DIMS, INV_LABEL_MAP, load_split_stage2
from src.evaluate import evaluate_stage2
from src.model import CLTStage2Model

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
CKPT_LOCAL = "outputs/roberta_stage2/best_model.pt"
CKPT_DRIVE = f"{DRIVE_OUTPUT_DIR}/roberta_stage2/best_model.pt"
CFG_LOCAL  = "outputs/roberta_stage2/config.yaml"
CFG_DRIVE  = f"{DRIVE_OUTPUT_DIR}/roberta_stage2/config.yaml"
OUT_LOCAL  = "outputs/roberta_stage2_test_eval"
OUT_DRIVE  = f"{DRIVE_OUTPUT_DIR}/roberta_stage2_test_eval"
TEST_PATH  = "data/processed/test.json"
MAX_LENGTH = 128
MODEL_NAME = "roberta-base"

# Resolve checkpoint
ckpt_path = CKPT_LOCAL if os.path.exists(CKPT_LOCAL) else CKPT_DRIVE
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f"RoBERTa Stage 2 checkpoint not found at {CKPT_LOCAL} or {CKPT_DRIVE}.\n"
        "Run Cell 8 first."
    )
print(f"Loading checkpoint: {ckpt_path}")

# Resolve model_name from saved config if available
cfg_path = CFG_LOCAL if os.path.exists(CFG_LOCAL) else (
    CFG_DRIVE if os.path.exists(CFG_DRIVE) else None
)
if cfg_path:
    import yaml
    with open(cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    MODEL_NAME = _cfg.get("model_name", MODEL_NAME)
print(f"Encoder base: {MODEL_NAME}")

os.makedirs(OUT_LOCAL, exist_ok=True)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = CLTStage2Model(MODEL_NAME).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

def tokenize_s2(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

test_ds       = load_split_stage2(TEST_PATH).map(tokenize_s2, batched=True)
test_ds_torch = test_ds.remove_columns(["text"])
test_ds_torch.set_format("torch")
print(f"Salient test sentences: {len(test_ds_torch)}")

collator = DataCollatorWithPadding(tokenizer)
loader   = DataLoader(test_ds_torch, batch_size=32, shuffle=False, collate_fn=collator)

all_preds = {d: [] for d in DIMS}
all_gold  = {d: [] for d in DIMS}
with torch.no_grad():
    for batch in loader:
        logits_dict = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
        )
        for dim in DIMS:
            pred_idx = torch.argmax(logits_dict[dim], dim=-1).cpu().tolist()
            gold_idx = batch[dim].tolist()
            all_preds[dim].extend(INV_LABEL_MAP[p] for p in pred_idx)
            all_gold[dim].extend(INV_LABEL_MAP[g] for g in gold_idx)

core = evaluate_stage2(all_preds, all_gold)

# Build serialisable metrics dict with per-class precision/recall added
metrics_out = {"mean_macro_f1": core["mean_macro_f1"]}
for dim in DIMS:
    dm        = core[dim]
    preds_arr = np.array(all_preds[dim])
    gold_arr  = np.array(all_gold[dim])
    per_class_precision = precision_score(
        gold_arr, preds_arr, average=None, labels=[-1, 0, 1], zero_division=0
    ).tolist()
    per_class_recall = recall_score(
        gold_arr, preds_arr, average=None, labels=[-1, 0, 1], zero_division=0
    ).tolist()
    metrics_out[dim] = {
        "macro_f1":            dm["macro_f1"],
        "per_class_precision": per_class_precision,
        "per_class_recall":    per_class_recall,
        "per_class_f1":        dm["per_class_f1"],
        "confusion_matrix":    dm["confusion_matrix"].tolist(),
        "krippendorff_alpha":  dm["krippendorff_alpha"],
        "collapsed":           dm["collapsed"],
    }

with open(f"{OUT_LOCAL}/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
with open(f"{OUT_LOCAL}/test_predictions.json", "w") as f:
    json.dump({"predictions": all_preds, "gold": all_gold}, f, indent=2)

# Confusion matrix PNGs — one per dimension
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    for dim in DIMS:
        cm   = core[dim]["confusion_matrix"]
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=["N/A", "Near", "Far"]
        )
        disp.plot(colorbar=False)
        plt.title(f"Test confusion matrix — RoBERTa Stage 2 {dim} (dev-trained)")
        plt.tight_layout()
        plt.savefig(f"{OUT_LOCAL}/confusion_matrix_{dim}.png", dpi=150)
        plt.close()
    print("Confusion matrix PNGs saved.")
except ImportError:
    print("matplotlib not installed — skipping confusion matrix PNGs")

# Back up to Drive
if os.path.isdir(OUT_DRIVE):
    shutil.rmtree(OUT_DRIVE)
shutil.copytree(OUT_LOCAL, OUT_DRIVE)

print("\n" + "=" * 60)
print("RoBERTa Stage 2 — TEST evaluation (dev-trained checkpoint)")
print("-" * 60)
print(f"  Mean macro-F1 : {core['mean_macro_f1']:.4f}")
for dim in DIMS:
    dm   = metrics_out[dim]
    note = "  [COLLAPSE]" if dm["collapsed"] else ""
    print(f"  {dim:<14}: macro-F1={dm['macro_f1']:.4f}  α={dm['krippendorff_alpha']:.4f}{note}")
print("=" * 60)
print(f"Outputs: {OUT_LOCAL}/  |  Drive: {OUT_DRIVE}/")


## Cell 12 — Combined Results Table

Loads `metrics.json` from all eight runs (four dev-shootout + four test-eval) and prints:

1. **Summary table** — Model | Stage | Dev macro-F1 | Test macro-F1 (Stage 2 per dimension).
2. **Winning-model diagnostic** — BERT Stage 1 test: full precision / recall / F1 /
   confusion matrix, labelled "Winning model — full diagnostic on test set".

In [ ]:
import json
import os

DRIVE_BASE = "/content/drive/MyDrive/CLT_Thesis/outputs"
DIMS = ("temporal", "spatial", "social", "hypothetical")


def _load_json(local_path, drive_path=None):
    if os.path.exists(local_path):
        with open(local_path) as f:
            return json.load(f)
    if drive_path and os.path.exists(drive_path):
        with open(drive_path) as f:
            return json.load(f)
    return None


def load_s1(run_name):
    return _load_json(
        f"outputs/{run_name}/metrics.json",
        f"{DRIVE_BASE}/{run_name}/metrics.json",
    )


def load_s2_best(run_name):
    """Test-eval runs write a single metrics.json; dev-shootout runs write epoch_metrics.json."""
    direct = _load_json(
        f"outputs/{run_name}/metrics.json",
        f"{DRIVE_BASE}/{run_name}/metrics.json",
    )
    if direct is not None and "mean_macro_f1" in direct:
        return direct
    epoch_data = _load_json(
        f"outputs/{run_name}/epoch_metrics.json",
        f"{DRIVE_BASE}/{run_name}/epoch_metrics.json",
    )
    if epoch_data is not None:
        return max(epoch_data, key=lambda r: r["mean_macro_f1"])
    return None


def fmt(val):
    return f"{val:.4f}" if val is not None else "  N/A  "


# Load all eight metric files
bert_s1_dev   = load_s1("bert_stage1")
bert_s2_dev   = load_s2_best("bert_stage2")
rob_s1_dev    = load_s1("roberta_stage1")
rob_s2_dev    = load_s2_best("roberta_stage2")
bert_s1_test  = load_s1("bert_stage1_test_eval")
bert_s2_test  = load_s2_best("bert_stage2_test_eval")
rob_s1_test   = load_s1("roberta_stage1_test_eval")
rob_s2_test   = load_s2_best("roberta_stage2_test_eval")

for label, obj in [
    ("bert_stage1 (dev)",         bert_s1_dev),
    ("bert_stage2 (dev)",         bert_s2_dev),
    ("roberta_stage1 (dev)",      rob_s1_dev),
    ("roberta_stage2 (dev)",      rob_s2_dev),
    ("bert_stage1_test_eval",     bert_s1_test),
    ("bert_stage2_test_eval",     bert_s2_test),
    ("roberta_stage1_test_eval",  rob_s1_test),
    ("roberta_stage2_test_eval",  rob_s2_test),
]:
    if obj is None:
        print(f"WARNING: metrics not found for {label} — run the corresponding cell first.")

# ── Summary table ─────────────────────────────────────────────────────────────
SEP = "=" * 75
print()
print(SEP)
print("SHOOTOUT CHECKPOINTS — Dev vs Test macro-F1 (dev-trained models)")
print(SEP)
HDR = f"{'Model':<14}  {'Stage':<20}  {'Dev macro-F1':>12}  {'Test macro-F1':>13}"
print(HDR)
print("-" * len(HDR))


def s1_row(model_label, dev_obj, test_obj):
    dev_f1  = dev_obj["macro_f1"]  if dev_obj  else None
    test_f1 = test_obj["macro_f1"] if test_obj else None
    print(f"{model_label:<14}  {'Stage 1 (salience)':<20}  {fmt(dev_f1):>12}  {fmt(test_f1):>13}")


def s2_rows(model_label, dev_obj, test_obj):
    for dim in DIMS:
        dev_f1   = dev_obj[dim]["macro_f1"]  if dev_obj  and dim in dev_obj  else None
        test_f1  = test_obj[dim]["macro_f1"] if test_obj and dim in test_obj else None
        dev_col  = dev_obj[dim].get("collapsed", False)  if dev_obj  and dim in dev_obj  else False
        test_col = test_obj[dim].get("collapsed", False) if test_obj and dim in test_obj else False
        dev_str  = fmt(dev_f1)  + (" [C]" if dev_col  else "")
        test_str = fmt(test_f1) + (" [C]" if test_col else "")
        print(f"{model_label:<14}  {f'Stage 2 — {dim}':<20}  {dev_str:>12}  {test_str:>13}")
    dev_mean  = dev_obj["mean_macro_f1"]  if dev_obj  else None
    test_mean = test_obj["mean_macro_f1"] if test_obj else None
    print(f"{model_label:<14}  {'Stage 2 — MEAN':<20}  {fmt(dev_mean):>12}  {fmt(test_mean):>13}")


s1_row("BERT-base",     bert_s1_dev,  bert_s1_test)
s2_rows("BERT-base",    bert_s2_dev,  bert_s2_test)
print("-" * len(HDR))
s1_row("RoBERTa-base",  rob_s1_dev,   rob_s1_test)
s2_rows("RoBERTa-base", rob_s2_dev,   rob_s2_test)
print(SEP)
print("[C] = model collapsed to predicting a single class for that dimension.")
print()

# ── Winning model full diagnostic ─────────────────────────────────────────────
print(SEP)
print("Winning model — full diagnostic on test set")
print("BERT-base Stage 1 (dev-trained shootout checkpoint, evaluated on test.json)")
print(SEP)
if bert_s1_test is None:
    print("  WARNING: bert_stage1_test_eval metrics not found — run Cell 11a first.")
else:
    m = bert_s1_test
    print(f"  Macro-F1    : {m['macro_f1']:.4f}")
    print(f"  Accuracy    : {m['accuracy']:.4f}")
    print(f"  Kripp. α    : {m['krippendorff_alpha']:.4f}")
    print()
    print("  Per-class metrics:")
    print(f"  {'Class':<14}  {'Precision':>9}  {'Recall':>9}  {'F1':>9}")
    print(f"  {'-'*14}  {'-'*9}  {'-'*9}  {'-'*9}")
    for i, lbl in enumerate(["not_salient", "salient"]):
        p = m["per_class_precision"][i]
        r = m["per_class_recall"][i]
        f = m["per_class_f1"][i]
        print(f"  {lbl:<14}  {p:>9.4f}  {r:>9.4f}  {f:>9.4f}")
    print()
    print("  Confusion matrix (rows=gold, cols=pred):")
    print(f"  {'':14}  {'pred: not_sal':>13}  {'pred: salient':>13}")
    for i, row_lbl in enumerate(["gold: not_sal", "gold: salient"]):
        row = m["confusion_matrix"][i]
        print(f"  {row_lbl:<14}  {row[0]:>13}  {row[1]:>13}")
print(SEP)


---

# Final Model Run — Post-Shootout (BERT only, train+dev → test)

BERT-base won the 9-classifier shootout. This section retrains BERT on **train+dev combined**
and evaluates **once** on the held-out test set.

**Rules:**
- Do not run these cells until the shootout model selection is final.
- The test set is touched exactly once — when Cells B and C evaluate the final model.
- Cells 1–10 (the original shootout) remain untouched and independently re-runnable.
- Run Final Cell D after training to restore the original data files so Cells 5–9 stay runnable.

**Implementation:** file-swap (fallback path) — `train.py` and `data.py` have hardcoded
`data/processed/train.json` / `data/processed/dev.json` paths with no CLI override flags,
so the final-model workflow temporarily replaces those files on disk for each training run
and restores them afterward. Original files are backed up as `*_original.json`.

| Final Cell | What it does |
|------------|--------------------------------------------------------------|
| A | Build combined train+dev, stage test.json as eval split |
| B | BERT Stage 1 final training (train+dev → test) |
| C | BERT Stage 2 final training (train+dev → test) |
| D | Restore original train.json / dev.json from backups |
| E | Final results summary — dev (shootout) vs test (final) |

## Final Cell A — Build Combined Train+Dev Data

Backs up original `train.json` and `dev.json`, concatenates train+dev into `train.json`,
and stages `test.json` as `dev.json` so the unmodified trainer reads the right files.

Expected counts after swap: **train=899** (combined), **eval=155** (test).
Idempotent — re-running detects the swap is already in place and skips it.

In [ ]:
import json
import os
import shutil

DATA_DIR   = "data/processed"
TRAIN_PATH = f"{DATA_DIR}/train.json"
DEV_PATH   = f"{DATA_DIR}/dev.json"
TEST_PATH  = f"{DATA_DIR}/test.json"
TRAIN_ORIG = f"{DATA_DIR}/train_original.json"
DEV_ORIG   = f"{DATA_DIR}/dev_original.json"

print("=" * 60)
print("IMPLEMENTATION PATH: fallback file-swap")
print("  train.py/data.py have hardcoded split paths with no CLI override.")
print("  Files are temporarily swapped on disk and restored via Final Cell D.")
print("=" * 60)

for path in [TRAIN_PATH, DEV_PATH, TEST_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")

backups_exist = os.path.exists(TRAIN_ORIG) and os.path.exists(DEV_ORIG)

if backups_exist:
    with open(TRAIN_PATH) as f:
        n_train_now = len(json.load(f))
    with open(DEV_PATH) as f:
        n_dev_now = len(json.load(f))
    if n_train_now == 899 and n_dev_now == 155:
        print(f"Swap already in place — train={n_train_now}, eval(dev slot)={n_dev_now}. Skipping.")
    else:
        print(f"WARNING: Backups exist but counts ({n_train_now}/{n_dev_now}) are unexpected.")
        print("Run Final Cell D to restore originals first, then re-run this cell.")
        raise RuntimeError("Unexpected state — restore originals first (Final Cell D).")
else:
    with open(TRAIN_PATH) as f:
        train_records = json.load(f)
    with open(DEV_PATH) as f:
        dev_records = json.load(f)
    with open(TEST_PATH) as f:
        test_records = json.load(f)

    assert len(train_records) == 723, f"train.json: expected 723 records, got {len(train_records)}"
    assert len(dev_records)   == 176, f"dev.json: expected 176 records, got {len(dev_records)}"
    assert len(test_records)  == 155, f"test.json: expected 155 records, got {len(test_records)}"
    print(f"Verified originals — train={len(train_records)}, dev={len(dev_records)}, test={len(test_records)}")

    shutil.copy2(TRAIN_PATH, TRAIN_ORIG)
    shutil.copy2(DEV_PATH, DEV_ORIG)
    print(f"Backed up: {TRAIN_ORIG}")
    print(f"Backed up: {DEV_ORIG}")

    combined = train_records + dev_records
    with open(TRAIN_PATH, "w", encoding="utf-8") as f:
        json.dump(combined, f)
    shutil.copy2(TEST_PATH, DEV_PATH)
    print(f"\nSwap complete:")
    print(f"  {TRAIN_PATH} <- train+dev combined ({len(combined)} records)")
    print(f"  {DEV_PATH}   <- test.json contents ({len(test_records)} records)")

with open(TRAIN_PATH) as f:
    n_final_train = len(json.load(f))
with open(DEV_PATH) as f:
    n_final_dev = len(json.load(f))
print(f"\nFinal counts — train (combined): {n_final_train}  |  eval (test slot): {n_final_dev}")
assert n_final_train == 899, f"Expected 899, got {n_final_train}"
assert n_final_dev   == 155, f"Expected 155, got {n_final_dev}"
print("\nFinal Cell A OK — data ready for final BERT training.")

## Final Cell B — BERT Stage 1 Final Training (train+dev → test)

Retrains `bert-base-uncased` for salience using the combined train+dev set.
Evaluates on the held-out test set. **Run Final Cell A first.**

Config: `configs/bert_stage1_final.yaml` — identical hyperparameters to the shootout.
Outputs saved to `outputs/bert_stage1_final/` and backed up to Drive.

> If training crashes, run **Final Cell D** immediately to restore data files.

In [ ]:
import json
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import yaml
from transformers import AutoTokenizer, DataCollatorWithPadding, TrainingArguments

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

# Import helpers directly from src — this lets us skip verify_splits(),
# which hard-checks train=723/dev=176/test=155 and would crash on the
# swapped combined-train file.
from src.data import load_split
from src.evaluate import compute_metrics_dict
from src.model import build_model
from src.train import WeightedTrainer, load_config, seed_everything

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
CFG_PATH     = "configs/bert_stage1_final.yaml"
RUN_NAME     = "bert_stage1_final"
LOCAL_OUT    = f"outputs/{RUN_NAME}"
DRIVE_OUT    = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"

print("=" * 60)
print("BERT Stage 1 FINAL — hyperparameters (same as shootout):")
print("  model      : bert-base-uncased")
print("  lr         : 2e-5")
print("  num_epochs : 5")
print("  batch_size : 16 (GPU) / 4 (CPU)")
print("  max_length : 128")
print("  seed       : 42")
print("  config     : configs/bert_stage1_final.yaml")
print("  train data : data/processed/train.json  (train+dev combined, n=899)")
print("  eval data  : data/processed/dev.json    (= test.json, n=155)")
print("  NOTE: training is inlined (not subprocess) so verify_splits() is")
print("        intentionally skipped — it hard-checks original split counts.")
print("=" * 60)

# Verify swap is in place before touching anything
with open("data/processed/train.json") as f:
    n_train = len(json.load(f))
with open("data/processed/dev.json") as f:
    n_dev = len(json.load(f))
if n_train != 899 or n_dev != 155:
    raise RuntimeError(
        f"Data swap not in place (train={n_train}, dev={n_dev}). Run Final Cell A first."
    )
print(f"Data verified — train={n_train} (combined), eval={n_dev} (test)")

cfg = load_config(CFG_PATH, stage=1)
seed_everything(cfg["seed"])

use_gpu    = torch.cuda.is_available()
batch_size = cfg["gpu_batch_size"] if use_gpu else cfg["cpu_batch_size"]
print(f"Device: {'cuda' if use_gpu else 'cpu'}  |  batch_size: {batch_size}  |  fp16: {use_gpu}")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 1 FINAL training started")

data_dir  = Path("data/processed")
tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True,
                     max_length=cfg["max_length"], padding=False)

train_ds = load_split(data_dir / "train.json").map(tokenize, batched=True)
dev_ds   = load_split(data_dir / "dev.json").map(tokenize, batched=True)
train_ds = train_ds.remove_columns(["text"])
dev_ds   = dev_ds.remove_columns(["text"])
train_ds.set_format("torch")
dev_ds.set_format("torch")
print(f"Stage 1 data — train: {len(train_ds)}, eval: {len(dev_ds)}")

# Class weights (same formula as train.py, computed from combined train only)
labels  = train_ds["label"]
n_total = len(labels)
n_pos   = int(sum(labels))
n_neg   = n_total - n_pos
w_neg   = n_total / (2 * n_neg)
w_pos   = n_total / (2 * n_pos)
class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32)
print(f"Class weights — 0 (not salient): {w_neg:.4f}, 1 (salient): {w_pos:.4f}")

model   = build_model(cfg["model_name"])
run_dir = Path(cfg["output_dir"])
run_dir.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(run_dir),
    num_train_epochs=cfg["num_epochs"],
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=float(cfg["lr"]),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=use_gpu,
    seed=cfg["seed"],
    report_to="none",
    run_name=cfg["run_name"],
)

def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    m = compute_metrics_dict(preds.tolist(), labels.tolist())
    return {"macro_f1": m["macro_f1"], "accuracy": m["accuracy"]}

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=hf_compute_metrics,
)

trainer.train()
shutil.copy(CFG_PATH, run_dir / "config.yaml")

preds_output = trainer.predict(dev_ds)
pred_labels  = np.argmax(preds_output.predictions, axis=-1).tolist()
gold_labels  = dev_ds["label"]
if hasattr(gold_labels, "tolist"):
    gold_labels = gold_labels.tolist()

metrics = compute_metrics_dict(pred_labels, gold_labels)

with open(run_dir / "dev_predictions.json", "w") as f:
    json.dump({"predictions": [int(x) for x in pred_labels],
               "gold":        [int(x) for x in gold_labels]}, f)

metrics_out = {k: (v.tolist() if hasattr(v, "tolist") else v)
               for k, v in metrics.items() if k != "confusion_matrix"}
metrics_out["confusion_matrix"] = metrics["confusion_matrix"].tolist()
with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    disp = ConfusionMatrixDisplay(confusion_matrix=metrics["confusion_matrix"],
                                  display_labels=["not salient", "salient"])
    disp.plot(colorbar=False)
    plt.title("Test confusion matrix — Stage 1 final")
    plt.tight_layout()
    plt.savefig(run_dir / "confusion_matrix.png", dpi=150)
    plt.close()
except ImportError:
    print("matplotlib not installed — skipping confusion matrix PNG")

t_end = datetime.now()
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {t_end - t_start})")

print("\n" + "=" * 60)
print(f"BERT Stage 1 — TEST macro-F1 (not dev): {metrics['macro_f1']:.4f}")
print(f"BERT Stage 1 — TEST accuracy           : {metrics['accuracy']:.4f}")
print("=" * 60)

# Back up to Drive
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)
n_files = sum(len(fls) for _, _, fls in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, fle))
    for root, _, fls in os.walk(DRIVE_OUT) for fle in fls
)
print(f"\nBERT Stage 1 FINAL saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")


## Final Cell C — BERT Stage 2 Final Training (train+dev → test)

Trains four parallel CLT heads on the combined train+dev salient sentences.
Evaluates on the held-out test set. **Run Final Cell A first.**

Config: `configs/bert_stage2_final.yaml` — identical hyperparameters to the shootout.
Outputs saved to `outputs/bert_stage2_final/` and backed up to Drive.

> If training crashes, run **Final Cell D** immediately to restore data files.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME     = "bert_stage2_final"
LOCAL_OUT    = f"outputs/{RUN_NAME}"
DRIVE_OUT    = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
METRICS_PATH = f"{LOCAL_OUT}/epoch_metrics.json"

print("=" * 60)
print("BERT Stage 2 FINAL — hyperparameters (same as shootout):")
print("  model        : bert-base-uncased")
print("  lr           : 2e-5")
print("  num_epochs   : 5")
print("  batch_size   : 16 (GPU) / 4 (CPU)")
print("  max_length   : 128")
print("  warmup_ratio : 0.1")
print("  seed         : 42")
print("  config       : configs/bert_stage2_final.yaml")
print("  train data   : data/processed/train.json  (train+dev salient, combined)")
print("  eval data    : data/processed/dev.json    (= test.json salient, n~127)")
print("=" * 60)

with open("data/processed/train.json") as f:
    n_train = len(json.load(f))
with open("data/processed/dev.json") as f:
    n_dev = len(json.load(f))
if n_train != 899 or n_dev != 155:
    raise RuntimeError(
        f"Data swap not in place (train={n_train}, dev={n_dev}). Run Final Cell A first."
    )
print(f"Data verified — train={n_train} (combined), eval={n_dev} (test)")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 2 FINAL training started")

try:
    subprocess.run(
        [sys.executable, "src/train.py",
         "--config", "configs/bert_stage2_final.yaml",
         "--stage", "2"],
        check=True,
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    print("Run Final Cell D to restore original data files before anything else.")
    raise

t_end = datetime.now()
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {t_end - t_start})")

with open(METRICS_PATH) as f:
    epoch_records = json.load(f)

best = max(epoch_records, key=lambda r: r["mean_macro_f1"])

print("\n" + "=" * 60)
print(f"BERT Stage 2 FINAL — TEST scores (not dev)  |  Best epoch: {best['epoch']}")
print("-" * 60)
for dim in ("temporal", "spatial", "social", "hypothetical"):
    f1 = best[dim]["macro_f1"]
    collapsed = best[dim].get("collapsed", False)
    note = "  [COLLAPSE]" if collapsed else ""
    print(f"  {dim:<14}: TEST macro-F1 = {f1:.4f}{note}")
print(f"  {'mean':<14}: TEST macro-F1 = {best['mean_macro_f1']:.4f}")
print("=" * 60)

if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)
n_files = sum(len(fls) for _, _, fls in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, fle))
    for root, _, fls in os.walk(DRIVE_OUT) for fle in fls
)
print(f"\nBERT Stage 2 FINAL saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Final Cell D — Restore Original Data Files

Restores `train.json` and `dev.json` from the `_original.json` backups created in Final Cell A.
Always safe to run — idempotent. Run this even if training in Cells B or C crashed.

After running this cell, Cells 5–9 (the shootout cells) are fully runnable again.

In [ ]:
import json
import os
import shutil

DATA_DIR   = "data/processed"
TRAIN_PATH = f"{DATA_DIR}/train.json"
DEV_PATH   = f"{DATA_DIR}/dev.json"
TRAIN_ORIG = f"{DATA_DIR}/train_original.json"
DEV_ORIG   = f"{DATA_DIR}/dev_original.json"

print("=" * 60)
print("Final Cell D — Restoring original data files")
print("=" * 60)

if not os.path.exists(TRAIN_ORIG) and not os.path.exists(DEV_ORIG):
    print("No backup files found — checking if originals are already in place.")
    with open(TRAIN_PATH) as f:
        n_train = len(json.load(f))
    with open(DEV_PATH) as f:
        n_dev = len(json.load(f))
    if n_train == 723 and n_dev == 176:
        print(f"Verified: train={n_train}, dev={n_dev} — originals already in place. Nothing to do.")
    else:
        print(f"WARNING: No backups and counts are unexpected (train={n_train}, dev={n_dev}).")
        print("Upload the original split files manually.")
else:
    restored = []
    if os.path.exists(TRAIN_ORIG):
        shutil.copy2(TRAIN_ORIG, TRAIN_PATH)
        restored.append("train.json")
    if os.path.exists(DEV_ORIG):
        shutil.copy2(DEV_ORIG, DEV_PATH)
        restored.append("dev.json")
    print(f"Restored: {restored}")

    with open(TRAIN_PATH) as f:
        n_train = len(json.load(f))
    with open(DEV_PATH) as f:
        n_dev = len(json.load(f))
    assert n_train == 723, f"Unexpected train count after restore: {n_train}"
    assert n_dev   == 176, f"Unexpected dev count after restore: {n_dev}"
    print(f"Verified restored counts — train={n_train}, dev={n_dev}")

print("\nFinal Cell D OK — shootout cells (5–9) are safe to re-run.")

## Final Cell E — Final Results Summary

Reads shootout dev scores from `outputs/bert_stage1/` and `outputs/bert_stage2/`
and test scores from `outputs/bert_stage1_final/` and `outputs/bert_stage2_final/`.

Prints a single table for the thesis: **Dev (shootout)** vs **Test (final)**.

In [ ]:
import json
import os

DIMS = ("temporal", "spatial", "social", "hypothetical")


def load_s1(run_name):
    for base in [f"outputs/{run_name}",
                 f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}"]:
        p = f"{base}/metrics.json"
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
    return None


def load_s2_best(run_name):
    for base in [f"outputs/{run_name}",
                 f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}"]:
        p = f"{base}/epoch_metrics.json"
        if os.path.exists(p):
            with open(p) as f:
                records = json.load(f)
            return max(records, key=lambda r: r["mean_macro_f1"])
    return None


def fmt(val):
    return f"{val:.4f}" if val is not None else "  N/A  "


s1_dev  = load_s1("bert_stage1")
s2_dev  = load_s2_best("bert_stage2")
s1_test = load_s1("bert_stage1_final")
s2_test = load_s2_best("bert_stage2_final")

for label, obj in [
    ("bert_stage1 (dev)",         s1_dev),
    ("bert_stage2 (dev)",         s2_dev),
    ("bert_stage1_final (test)",  s1_test),
    ("bert_stage2_final (test)",  s2_test),
]:
    if obj is None:
        print(f"WARNING: metrics not found for {label}")

print("=" * 60)
print("BERT-base Stage 1 — salience macro-F1")
print("-" * 60)
print(f"  Dev  (shootout) : {fmt(s1_dev['macro_f1']  if s1_dev  else None)}")
print(f"  Test (final)    : {fmt(s1_test['macro_f1'] if s1_test else None)}")
print()

print("BERT-base Stage 2 — CLT dimension macro-F1")
print("-" * 60)
col_w = 14
print(f"  {'Dimension':<{col_w}}  {'Dev (shootout)':>16}  {'Test (final)':>14}")
print("  " + "-" * (col_w + 34))
for dim in DIMS:
    dev_f1   = s2_dev[dim]["macro_f1"]  if s2_dev  else None
    test_f1  = s2_test[dim]["macro_f1"] if s2_test else None
    dev_col  = s2_dev[dim].get("collapsed", False)  if s2_dev  else False
    test_col = s2_test[dim].get("collapsed", False) if s2_test else False
    dev_note  = " [C]" if dev_col  else ""
    test_note = " [C]" if test_col else ""
    print(f"  {dim:<{col_w}}  {fmt(dev_f1) + dev_note:>16}  {fmt(test_f1) + test_note:>14}")

dev_mean  = s2_dev["mean_macro_f1"]  if s2_dev  else None
test_mean = s2_test["mean_macro_f1"] if s2_test else None
print("  " + "-" * (col_w + 34))
print(f"  {'mean':<{col_w}}  {fmt(dev_mean):>16}  {fmt(test_mean):>14}")
print("=" * 60)
print("[C] = model collapsed to predicting a single class for that dimension.")
print()
print("Note: Dev scores are from the 9-classifier shootout (train -> dev).")
print("      Test scores are from the final BERT run (train+dev -> test).")
print("      Test set was touched exactly once.")